In [ ]:
%%capture
!pip install torch==2.4.0 torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.45.2 tokenizers==0.20.3
!pip install accelerate==0.34.0 bitsandbytes==0.43.0
!pip install peft==0.14.0 datasets pillow
!pip install jiwer einops addict easydict

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import sys, os
from huggingface_hub import snapshot_download

MODEL_DIR = "./deepseek_ocr2"
snapshot_download("deepseek-ai/DeepSeek-OCR-2", local_dir=MODEL_DIR, force_download=True)
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)
print("Model downloaded")

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
    # device_map HATA DO
    trust_remote_code=True,
    use_safetensors=True,
)
model = model.to("cuda:0")  # Explicit cuda:0

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    use_fast=True,
)

print("✅ Model & Tokenizer loaded!")
print(f"Dtype: {model.dtype}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    use_rslora=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training ke liye zaroori
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
from datasets import load_dataset
from PIL import Image
import os

DATASET_DIR = "/kaggle/input/datasets/zahidazam/deepseek-ocr-2-finetuning-dataset"

print("Dataset dir exists:", os.path.exists(DATASET_DIR))
if os.path.exists(DATASET_DIR):
    print("Files:", os.listdir(DATASET_DIR))

dataset = load_dataset("json", data_files={
    "train": f"{DATASET_DIR}/train.jsonl",
    "validation": f"{DATASET_DIR}/val.jsonl", 
    "test": f"{DATASET_DIR}/test.jsonl",
})

print(f"\nTrain: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")
print("Sample:", dataset['train'][0])

In [ ]:
from PIL import Image

def convert_user_format(sample):
    conversation = []
    images = []
    
    for msg in sample["messages"]:
        if msg["role"] == "user":
            text_content = None
            for part in msg["content"]:
                if part["type"] == "image":
                    # Relative path ko absolute banao
                    img_path = os.path.join(DATASET_DIR, part["image"])
                    img = Image.open(img_path).convert("RGB")
                    images.append(img)
                elif part["type"] == "text":
                    text_content = part["text"]
            
            conversation.append({
                "role": "<|User|>",
                "content": text_content,
                "images": images
            })
        elif msg["role"] == "assistant":
            conversation.append({
                "role": "<|Assistant|>",
                "content": msg["content"]
            })
    
    return {"messages": conversation}

# Convert all splits
converted_train = [convert_user_format(s) for s in dataset["train"]]
converted_val = [convert_user_format(s) for s in dataset["validation"]]
converted_test = [convert_user_format(s) for s in dataset["test"]]

print(f"✅ Converted: Train={len(converted_train)}, Val={len(converted_val)}, Test={len(converted_test)}")
print("Sample user text:", converted_train[0]["messages"][0]["content"])
print("Sample assistant text:", converted_train[0]["messages"][1]["content"][:50])

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/khowar_ocr_outputs",
    per_device_train_batch_size=1,      # T4 pe safe
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      # Effective batch = 8
    num_train_epochs=3,                 # 16K images ke liye
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.001,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
)

print("✅ Training arguments set")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total train steps: ~{len(converted_train) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

In [ ]:
print(f"Pad token: {tokenizer.pad_token}")
print(f"Pad token ID: {tokenizer.pad_token_id}")
print(f"Vocab size: {len(tokenizer)}")
print(f"BOS token ID: {tokenizer.bos_token_id}")
print(f"EOS token ID: {tokenizer.eos_token_id}")

# Agar pad_token_id None hai toh fix karo
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"Fixed pad_token_id to: {tokenizer.pad_token_id}")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

from transformers import Trainer

# DataParallel unwrap agar laga hai
if hasattr(model, "module"):
    model = model.module

data_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=converted_train,
    eval_dataset=converted_val,
    args=training_args,
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB reserved.")

trainer_stats = trainer.train()